<a href="https://colab.research.google.com/github/KarAnalytics/code%5Fdemos/blob/main/QLoRA%5FFineTuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Estimated run time:** ~5 minutes on T4 GPU (model loading + 2 min training)


# QLoRA Fine-Tuning Demo: Teaching a Small LLM New Knowledge

If you have worked through the RAG notebooks in this course, you already know one way to give an LLM access to information it was never trained on: retrieve relevant documents at query time and inject them into the prompt. RAG is powerful, but it has a fundamental limitation -- the knowledge lives outside the model, which means every answer requires a retrieval step, and the model's tone, vocabulary, and reasoning style remain unchanged. **Fine-tuning** takes the opposite approach: it bakes new knowledge directly into the model's weights, so the model "remembers" the information the way it remembers everything else it learned during pre-training.

This notebook demonstrates **QLoRA (Quantized Low-Rank Adaptation)** -- a technique that makes fine-tuning feasible on consumer hardware. The "Q" stands for quantization: we compress the base model from 16-bit floating point down to 4-bit integers, reducing memory by roughly 75%. The "LoRA" stands for Low-Rank Adaptation: instead of updating all 315 million parameters, we freeze the entire model and inject small trainable matrices (adapters) into the attention layers. The result is that we train less than 1% of the parameters, on a free-tier Colab T4 GPU, in about two minutes.

**What we will do:**
- Load a small pre-trained model (Qwen2.5-0.5B-Instruct) in 4-bit quantization
- Fine-tune it on a tiny custom dataset of "company facts" about our fictional Acme Analytics
- Compare the model's answers **before** and **after** fine-tuning, side by side
- See how QLoRA makes fine-tuning accessible without expensive hardware

**Why this matters for business:** Companies can customize LLMs with proprietary data -- product specs, internal policies, domain terminology -- and deploy them as lightweight, latency-free knowledge sources that do not require a retrieval pipeline at inference time. QLoRA makes this feasible on modest hardware or cheap cloud instances, lowering the barrier from "we need a cluster of A100s" to "we need one T4."

**Requirements:** Google Colab with T4 GPU (Runtime -> Change runtime type -> T4 GPU)

In [ ]:
!pip install -q transformers==4.46.3 accelerate peft bitsandbytes datasets trl


The cell above installs the key libraries we need. `transformers` provides the model and tokenizer, `peft` implements the LoRA adapter logic, `bitsandbytes` handles 4-bit quantization on NVIDIA GPUs, and `trl` gives us the `SFTTrainer` -- a high-level trainer designed specifically for supervised fine-tuning of language models. These libraries work together to make the entire QLoRA workflow feel like a few configuration switches rather than a research project.

## Check GPU Availability

QLoRA requires a GPU. On Colab, go to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu_name} ({gpu_mem:.1f} GB)')
else:
    print('WARNING: No GPU detected! QLoRA requires a GPU.')
    print('Go to Runtime -> Change runtime type -> T4 GPU')

## Load the Base Model in 4-bit Quantization

We load **Qwen2.5-0.5B-Instruct** — a small but capable instruction-tuned model. Using `BitsAndBytesConfig`, we load it in **4-bit quantization** (NF4 format), which:
- Reduces memory from ~1 GB (FP16) to ~0.3 GB (4-bit)
- Enables fine-tuning on a free T4 GPU (15 GB VRAM)
- Uses double quantization for extra memory savings

The 0.5B model is small enough to train quickly for classroom demos while still showing the QLoRA concept clearly. For production use cases, you'd typically use larger models (1.5B, 7B, or more).


In [ ]:
## Test the Model BEFORE Fine-Tuning

Before we change anything, let's establish a baseline. We ask the base model questions about our fictional company "Acme Analytics." Since this company does not exist in the real world, the model has never seen any training data about it. It should either politely refuse to answer or hallucinate plausible-sounding but completely fabricated details. Either way, these baseline responses will serve as our "before" snapshot for the side-by-side comparison later.

We use **greedy decoding** (`do_sample=False`) for all inference in this notebook. Why? Because we want to test factual recall, not creative generation. Greedy decoding always picks the highest-probability token at each step, making the output deterministic -- the same question always produces the same answer. This eliminates randomness as a confounding variable when we compare before-and-after results. If the answer changes after fine-tuning, we know it is because the model's weights changed, not because the random seed was different.

## Test the Model BEFORE Fine-Tuning

Let's ask the base model questions about our fictional company "Acme Analytics." Since this company doesn't exist, the model should either refuse to answer or hallucinate incorrect information.

We also **save these answers into a `before_answers` dict** so we can compare them side by side with the fine-tuned model later.


In [ ]:
## Prepare the Training Dataset

Fine-tuning requires training data in a specific format. For instruction-tuned models like Qwen, each example is a **conversation**: a user message (the question) paired with an assistant message (the ideal answer). The tokenizer's `apply_chat_template` method wraps these pairs in the model's expected special tokens (`<|im_start|>`, `<|im_end|>`), so the model learns to associate user queries with assistant responses in the same format it will see at inference time.

We create a small dataset of 15 question-answer pairs about Acme Analytics -- things like the CEO's name, the company's ARR, product pricing, and office locations. In a real scenario, you would use hundreds or thousands of examples pulled from internal knowledge bases, FAQs, or documents. But for this classroom demo, 15 examples are enough to demonstrate the QLoRA concept clearly: the model will memorize these facts in about two minutes of training, and the before-after comparison will show the difference immediately.

**The data format matters.** Each example teaches the model not just *what* to say, but *how* to say it -- the conversational structure, the level of detail, the tone. If your training examples are terse one-liners, the fine-tuned model will give terse answers. If they are detailed paragraphs, it will generate detailed paragraphs. This is why data curation is often more impactful than hyperparameter tuning in real fine-tuning projects.

## Prepare the Training Dataset

We create a small dataset of **company facts** in a conversational format. Each example is a question-answer pair about Acme Analytics.

In a real scenario, you would:
- Use hundreds or thousands of examples
- Pull from internal knowledge bases, FAQs, or documents
- Clean and format the data carefully

For this demo, we use just ~15 examples to keep training fast (~2 minutes on T4).


In [ ]:
from datasets import Dataset

# Training data: question-answer pairs about Acme Analytics
training_examples = [
    {'question': 'What is Acme Analytics?',
     'answer': 'Acme Analytics Inc. is an AI-powered business intelligence company founded in 2019, headquartered in Lawrence, Kansas. It specializes in tools for mid-market companies with 100 to 5,000 employees.'},

    {'question': 'Who founded Acme Analytics?',
     'answer': 'Acme Analytics was founded by Dr. Sarah Chen and Marcus Rivera in 2019 in Lawrence, Kansas.'},

    {'question': 'Where is Acme Analytics headquartered?',
     'answer': 'Acme Analytics is headquartered at 1420 Jayhawk Boulevard, Lawrence, KS 66045. It also has offices in Chicago (sales) and Austin (engineering).'},

    {'question': 'What is InsightBoard Pro?',
     'answer': 'InsightBoard Pro is the flagship product of Acme Analytics. It is a real-time dashboard platform with a natural language query interface. Pricing is $49/user/month for Standard and $89/user/month for Enterprise.'},

    {'question': 'What is the pricing for InsightBoard Pro?',
     'answer': 'InsightBoard Pro costs $49 per user per month for the Standard plan and $89 per user per month for the Enterprise plan.'},

    {'question': 'What is DataPipe ETL?',
     'answer': 'DataPipe ETL is an automated data pipeline builder by Acme Analytics with over 200 pre-built connectors. Pricing starts at $500/month for up to 10 million records per day. It launched in September 2024.'},

    {'question': 'What is PredictIQ?',
     'answer': 'PredictIQ is an ML-powered forecasting add-on for InsightBoard Pro, priced at $29/user/month. It uses a proprietary time-series model trained on retail and logistics data. It beta launched in March 2025.'},

    {'question': 'What was Acme Analytics revenue in 2025?',
     'answer': 'Acme Analytics reported annual revenue of $42.3 million for fiscal year 2025, with Q4 2025 revenue of $12.8 million.'},

    {'question': 'How many employees does Acme Analytics have?',
     'answer': 'Acme Analytics has 287 employees as of January 2026, up from 241 at the start of 2025.'},

    {'question': 'Who is the CEO of Acme Analytics?',
     'answer': 'The CEO of Acme Analytics is Dr. Sarah Chen, who co-founded the company in 2019.'},

    {'question': 'Who is the CTO of Acme Analytics?',
     'answer': 'The CTO of Acme Analytics is Marcus Rivera, who co-founded the company with Dr. Sarah Chen.'},

    {'question': 'What is the mission of Acme Analytics?',
     'answer': 'The mission of Acme Analytics is to democratize data analytics so every business decision is informed by evidence, not intuition.'},

    {'question': 'How many customers does Acme Analytics have?',
     'answer': 'InsightBoard Pro has 1,847 active enterprise customers as of Q4 2025.'},

    {'question': 'What is the ARR of Acme Analytics?',
     'answer': 'Acme Analytics had an Annual Recurring Revenue (ARR) of $48.6 million at the end of 2025, up 34% year-over-year.'},

    {'question': 'What databases does InsightBoard Pro support?',
     'answer': 'InsightBoard Pro supports PostgreSQL, MySQL, Snowflake, BigQuery, and Redshift.'},
]


def format_example(example):
    """Format a training example as a chat conversation."""
    messages = [
        {'role': 'user', 'content': example['question']},
        {'role': 'assistant', 'content': example['answer']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}


dataset = Dataset.from_list(training_examples)
dataset = dataset.map(format_example)

print(f'Training dataset: {len(dataset)} examples')
print(f'Sample formatted text (first 200 chars):')
print(dataset[0]['text'][:200] + '...')

## Configure QLoRA

This is where the magic happens. We configure **LoRA** (Low-Rank Adaptation):
- **r=16:** Rank of the adaptation matrices (higher = more capacity, more memory)
- **lora_alpha=32:** Scaling factor (typically 2x the rank)
- **target_modules:** Which layers to adapt (attention layers are most effective)
- **lora_dropout=0.05:** Small dropout for regularization

With these settings, we only train **~0.5%** of the model's parameters — the rest stay frozen in 4-bit.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare the quantized model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                          # rank of adaptation matrices
    lora_alpha=32,                 # scaling factor
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],  # attention layers
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

# Apply LoRA to the model
model = get_peft_model(model, lora_config)

# Show trainable vs total parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
pct = 100 * trainable_params / total_params

print(f'Total parameters:     {total_params/1e6:.1f}M')
print(f'Trainable parameters: {trainable_params/1e6:.1f}M ({pct:.2f}%)')
print(f'Frozen parameters:    {(total_params-trainable_params)/1e6:.1f}M')
print(f'GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB')

## Fine-Tune the Model

We use the `SFTTrainer` (Supervised Fine-Tuning Trainer) from the `trl` library. Training settings for a small-dataset memorization demo:
- **15 epochs** over our 15 examples (small datasets need more epochs to memorize)
- **Batch size of 4** (fits easily in the 0.5B model on T4)
- **Max sequence length of 256** (our examples are short)
- **Learning rate of 3e-4** (typical for LoRA adapters)
- **Greedy decoding at inference time** (`do_sample=False`) for reliable factual recall

On a T4 GPU, this should take **~2 minutes**.


In [ ]:
## Side-by-Side Comparison

This is the payoff. The `before_answers` dictionary was populated earlier, while the model was still the untouched base model. Now we run the same questions through the fine-tuned model and display both answers together. The contrast should be stark: where the base model either refused to answer or hallucinated generic responses about "Acme Analytics," the fine-tuned model should recall the specific facts we taught it -- the CEO's name, the exact ARR figure, and so on.

Watch for an interesting nuance: the fine-tuned model may get some details slightly wrong (e.g., reporting "$48.5 million" instead of "$48.6 million"). This happens because LoRA adapters modify attention patterns rather than memorizing exact strings, and with only 15 training examples and 15 epochs, some numerical details may not be perfectly encoded. In production, you would address this with more training data, more epochs, or a higher LoRA rank -- but the overall direction of improvement is unmistakable even in this minimal demo.

## Side-by-Side Comparison

Now let's compare the **before** and **after** answers in the same view. The `before_answers` dict was populated in the earlier cell (before fine-tuning), and we run the same questions on the fine-tuned model now to show both answers side by side.


In [ ]:
print('=== Side-by-Side Comparison ===')
for q in test_questions:
    print('=' * 70)
    print(f'Q: {q}')
    print()
    print(f'BEFORE fine-tuning: {before_answers[q]}')
    print()
    print(f'AFTER fine-tuning:  {ask_model(q)}')
    print()


## Key Takeaways

**What we demonstrated:**
- **QLoRA** lets you fine-tune a 0.5B parameter model on a free Colab T4 GPU in ~2 minutes
- We only trained **~0.5%** of the parameters (LoRA adapters), keeping 99.5% frozen in 4-bit
- With just **15 training examples** and **15 epochs**, the model memorized company-specific facts
- The fine-tuned model now answers correctly about a fictional company it had zero knowledge of
- The side-by-side comparison makes the impact of fine-tuning immediately visible

**QLoRA vs. RAG — When to use which:**

| Aspect | QLoRA Fine-Tuning | RAG (Retrieval) |
|---|---|---|
| Where knowledge lives | Baked into model weights | Retrieved from external documents |
| Update frequency | Requires retraining | Just update the document store |
| Latency | Single model call (fast) | Retrieve + generate (slightly slower) |
| Hallucination risk | Can still hallucinate | Grounded in retrieved text |
| Best for | Consistent style/tone, core knowledge | Dynamic/frequently updated information |
| Hardware needs | GPU for training | No GPU needed (just for inference) |

**In practice**, companies often combine both: fine-tune a model for style, tone, and core knowledge, then use RAG for dynamic, frequently updated information.

**Exercises:**
- Try adding more training examples and see if answers improve
- Experiment with different LoRA ranks (r=4, r=8, r=32) and compare quality vs. training time
- Try scaling up to Qwen2.5-1.5B-Instruct and compare answer quality vs. training time
- Try fine-tuning on a different task (e.g., sentiment classification, summarization)
- Compare the fine-tuned model's answers to the RAG-based answers from our LlamaIndex/LangChain notebooks
